# MESA to Phlegethon


## Imports and Library Loading


In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from phleos import *
import importlib.util
from scipy.interpolate import interp1d
from scipy.integrate import cumulative_trapezoid
import scipy


## Physical Constants


In [ ]:
R_SOL = 6.957e10
c = 2.99792458e10
a = 7.5657e-15
R_GAS = 83144621.0

## Toggles and Input Values


In [ ]:
# =========================
# Runtime and plotting mode
# =========================
configure_plot_backend = True
plot_backend_mode = 'auto'  # 'auto', 'interactive', or 'inline'

# =================
# EOS / source data
# =================
#EOS settings
#table parameters
NRHO = 541
NT = 201
LOGRHOMIN = -12.0
LOGRHOMAX = 15.0
LOGTMIN = 3.0
LOGTMAX = 13.0

eos_mode = ['ions','radiation','elepos','coulomb']
helm_table_path = '/home/vitovskn/phlegethon/data/helm_table_timmes_x2.dat'

#paths
mesa_path = './mesa_profile/profile40.data'

hse_library_pointer = '/home/vitovskn/phlegethon/miscellaneous/create_input_library/hse_reintegration.py'
mesa_reader_pointer = '/home/vitovskn/phlegethon/miscellaneous/create_input_library/mesa_minireader.py'

# Requested MESA fields
profile_names = [
    'r', 'rho', 'grav', 'Abar', 'Zbar', 'P', 'T', 'kappa', 'gradT_sub_grada',
    's', 'gamma1', 'gamma3', 'cp', 'cv', 'energy', 'chiRho', 'grada',
    'conv_vel', 'Mach', 'sound', 'luminosity', 'eps_nuc',
]

# Optional alias overrides for naming differences
profile_name_aliases = {
    # 'gradT_sub_grada': ['gradT_sub_grada', 'superad'],
    # 'sound': ['csound', 'sound_speed'],
}
missing_profile_policy = 'warn'  # 'ignore', 'warn', or 'error'

# =======================================
# MESA pre-reintegration modifications
# =======================================
smoothing_width = -1
smoothing_kernel = 'bump'  # 'box', 'gaussian', or 'bump'

#Optional clamp applies only when integration_mode='nabladif_given'.
# Set to None to disable this clamp entirely.
nabladif_zero_tol = 1e-3

# ====================
# Abundance transition
# ====================
enable_abundance_transition = False
abundance_target_species = ['h1', 'he4', 'c12', 'o16']
abundance_closure_species = 'he4'  # must be in abundance_target_species
abundance_missing_fill = 0.0

# ========================
# Optional eps_nuc tweaking
# ========================
enable_enuc_tweaking = False
enuc_shift_dr = 0.0 * R_SOL  
enuc_renorm_factor = 1.0
enuc_plot_compare = True

# =========================
# Reintegration grid/domain
# =========================
Nr = 5000
rmin = 0.05 * R_SOL
rmax = 15.0 * R_SOL
RESIZE_FACTOR = 1

# =====================
# Integration selection
# =====================
integration_mode = 'nabladif_given'  # 'nabladif_given' or 't_given'
run_reintegration = True
use_eos_p0 = False
use_non_uniform_composition = True
uniform_composition_strategy = 'mean'  # 'mean' or 'median' used only when use_non_uniform_composition=False

# Anchor placement controls
anchor_radius_mode = 'fraction'  # 'fraction', 'absolute', or 'none'
r_start_frac = 0.5
r_start_abs = None
anchor_policy = 'nearest'  # 'nearest', 'clip', or 'strict'
eos_output_dtype = 'float64' 

# ==============
# Output controls
# ==============
output_file = False
name_of_in_file = 'TEST.in'
enuc_at_output = True
output_column_names_requested = [
    'r_cm', 'g_cms2', 'phi_ergg', 'P_cgs','rho_cgs', 'T_K', 'kappa_cgs',
    'Ye', 'inv_Abar', 'enuc_erg_cm3_s', 'X_h1', 'X_he4', 'X_c12', 'X_o16',
]

# =======================
# Plot and figure controls
# =======================
plot_title_fontsize = 12
plot_legend_fontsize = 9
plot_label_fontsize = 10
plot_axis_values_fontsize = 9
rphl_min = 0.15 * R_SOL
rphl_max = 4.0 * R_SOL
plot_every = int(Nr / 100)
entropy_alignment_mode = 'none'  # 'none' or 'anchor'
primary_keys = ['temperature', 'nabladiff', 'density', 'pressure', 'entropy', 'opacity', 'gravity']
figure1_scale_mode = 'split'  # 'split', 'linear', or 'log'
figure2_show_legends = False

## Setup and Profile Loading


In [ ]:
# Load hydrostatic reintegration backend from configured pointer
spec_hse = importlib.util.spec_from_file_location('hse_reintegration', hse_library_pointer)
hse = importlib.util.module_from_spec(spec_hse)
spec_hse.loader.exec_module(hse)

# Configure Matplotlib backend from toggle values
backend_info = hse.configure_plot_backend(
    mode=plot_backend_mode,
    enabled=configure_plot_backend,
    verbose=True,
)

eos_table = eos_fort.eos_fort_mod.load_table('%shelm_table_timmes_x2.dat'%(data),NRHO,NT,LOGRHOMIN,LOGRHOMAX,LOGTMIN,LOGTMAX)
print('phleos loaded!')

# Derive runtime settings from toggles (backend helper)
runtime_cfg = hse.derive_runtime_settings(
    enable_abundance_transition=enable_abundance_transition,
    abundance_target_species=abundance_target_species,
    abundance_closure_species=abundance_closure_species,
    anchor_radius_mode=anchor_radius_mode,
    rmin=rmin,
    rmax=rmax,
    r_start_frac=r_start_frac,
    r_start_abs=r_start_abs,
    use_non_uniform_composition=use_non_uniform_composition,
    uniform_composition_strategy=uniform_composition_strategy,
)
abundance_mode = runtime_cfg['abundance_mode']
abundance_names = runtime_cfg['abundance_names']
r_start = runtime_cfg['r_start']
unif_comp_mode = runtime_cfg['unif_comp_mode']

# Load local minimal MESA reader module
spec_mesa = importlib.util.spec_from_file_location('mesa_minireader', mesa_reader_pointer)
mesa_reader = importlib.util.module_from_spec(spec_mesa)
spec_mesa.loader.exec_module(mesa_reader)

# Load selected MESA profile fields
mmodel = mesa_reader.load_mesa_profile(
    filename=mesa_path,
    profile_names=profile_names,
    aliases=profile_name_aliases,
    missing_policy=missing_profile_policy,
    reverse_radius_order=True,
    abundance_mode=abundance_mode,
    abundance_names=abundance_names,
)

# Construct integration/plotting helpers
integrator = hse.HSEIntegrator(eos_table=eos_table, eos_mode=eos_mode)
plotter = hse.HSEPlotter(
    R_SOL,
    rmin=rmin,
    rmax=rmax,
    rphl_min=rphl_min,
    rphl_max=rphl_max,
    r_integration_start=r_start,
    title_fontsize=plot_title_fontsize,
    legend_fontsize=plot_legend_fontsize,
    label_fontsize=plot_label_fontsize,
    axis_value_fontsize=plot_axis_values_fontsize,
)

print('Loaded hydrostatic reintegration library + plotter')
print('Used backend:', hse_library_pointer)
print('Used mesa reader:', mesa_reader_pointer)
print('r-range [cm]:', rmin, 'to', rmax)
print('requested r_start [cm]:', r_start)
print('anchor_policy:', anchor_policy)
print('run_reintegration:', run_reintegration)
print('missing_profile_policy:', missing_profile_policy)
print('enable_abundance_transition:', enable_abundance_transition)
if enable_abundance_transition:
    print('abundance_target_species:', abundance_target_species)
    print('abundance_closure_species:', abundance_closure_species)
print('enable_enuc_tweaking:', enable_enuc_tweaking)
if enable_enuc_tweaking:
    print('enuc_shift_dr [cm]:', enuc_shift_dr)
    print('enuc_renorm_factor:', enuc_renorm_factor)
print('nabladif_zero_tol:', nabladif_zero_tol)
print('To set in app.F90 -> rphl_min:', rphl_min, 'rphl_max:', rphl_max)


## MESA Profile Extraction and Derived Diagnostics


In [ ]:
# Load MESA profiles from the unified profile list
r_mm = mmodel['r']
rho_mm = mmodel['rho']
gr_mm = mmodel['grav']
abar_mm = mmodel['Abar']
zbar_mm = mmodel['Zbar']
press_mm = mmodel['P']
temp_mm = mmodel['T']
kappa_mm = mmodel['kappa']
nabladif_mm = mmodel['gradT_sub_grada']

# Diagnostic fields are loaded identically and default to NaN when absent
def _profile(name):
    return mmodel.get(name, np.full_like(r_mm, np.nan, dtype=float))

entropy_mm = _profile('s')*R_GAS
gamma1_mm = _profile('gamma1')
gamma3_mm = _profile('gamma3')
cp_mm = _profile('cp')
cv_mm = _profile('cv')
energy_mm = _profile('energy')
nabla_ad_mm = _profile('grada')
conv_vel_mm = _profile('conv_vel')
cs_mm = _profile('sound')
luminosity_mm = _profile('luminosity')
eps_nuc_mm = _profile('eps_nuc')
chiRho_mm = _profile('chiRho')
mach_mm = conv_vel_mm / cs_mm

abundance_species_order = []
abundance_profiles_mm = {}
ye_mm = zbar_mm / np.maximum(abar_mm, 1e-99)

# Derived diagnostics with graceful fallbacks
alpha_mm = np.where(np.isfinite(chiRho_mm) & (chiRho_mm != 0.0), 1.0/chiRho_mm, np.nan)
delta_mm = np.sqrt(rho_mm*temp_mm*alpha_mm*(cp_mm - cv_mm)/press_mm)
nabla_mm = nabladif_mm + nabla_ad_mm
enuc_mm_original = np.where(np.isfinite(eps_nuc_mm), eps_nuc_mm * rho_mm, 0.0)
enuc_mm_modified = enuc_mm_original.copy()
enuc_mm = enuc_mm_original.copy()

# Respace to equal very fine grid
NrRESIZE = Nr*RESIZE_FACTOR
r_mme = hse.grid2grid(r_mm, r_mm, rmin, rmax, NrRESIZE)
gr_mme = hse.grid2grid(r_mm, gr_mm, rmin, rmax, NrRESIZE)
abar_mme = hse.grid2grid(r_mm, abar_mm, rmin, rmax, NrRESIZE)
zbar_mme = hse.grid2grid(r_mm, zbar_mm, rmin, rmax, NrRESIZE)
press_mme = hse.grid2grid(r_mm, press_mm, rmin, rmax, NrRESIZE)
temp_mme = hse.grid2grid(r_mm, temp_mm, rmin, rmax, NrRESIZE)
rho_mme = hse.grid2grid(r_mm, rho_mm, rmin, rmax, NrRESIZE)
nabladif_mme = hse.grid2grid(r_mm, nabladif_mm, rmin, rmax, NrRESIZE)

kappa = hse.grid2grid(r_mm, kappa_mm, rmin, rmax, Nr)

loaded_profiles = sorted(list(mmodel.resolved.keys()))
requested_optional_attrs = set(profile_names)
loaded_optional = list(loaded_profiles)
missing_optional_attrs = sorted(requested_optional_attrs - set(loaded_optional))

# MESA-only convective diagnostics (linear + log) via backend helper
conv_diag_mask = (r_mm > rmin) & (r_mm < rmax)
r_conv = r_mm[conv_diag_mask]
conv_vel_conv = conv_vel_mm[conv_diag_mask]
mach_conv = mach_mm[conv_diag_mask]
plotter.plot_conv_vel_mach(
    r_conv,
    conv_vel_conv,
    mach_conv,
    xlim=(rmin / R_SOL, rmax / R_SOL),
)
print('Mesa model read!')
print('Loaded profiles:', loaded_profiles)

In [ ]:
# frad, ftot and fconv
gradp_mm = (press_mm[2:] - press_mm[:-2])/(r_mm[2:] - r_mm[:-2])
gradp_mm = np.append(gradp_mm, gradp_mm[-1])
gradp_mm = np.insert(gradp_mm, 0, gradp_mm[0])
Hp_mm = -press_mm/(rho_mm*gr_mm)


f_rad = 4*a*c*temp_mm**4*nabla_mm/(3*kappa_mm*rho_mm*Hp_mm)
f_tot = luminosity_mm/(4*np.pi*r_mm**2)
f_conv = f_tot - f_rad

f_rad = f_rad[(r_mm < rmax) & (r_mm > rmin)]
f_tot = f_tot[(r_mm < rmax) & (r_mm > rmin)]
f_conv = f_conv[(r_mm < rmax) & (r_mm > rmin)]
r_mm_f = r_mm[(r_mm < rmax) & (r_mm > rmin)]

plotter.compare_flux_components(r_mm_f, f_rad, f_tot, f_conv)

## Abundance Transition


In [ ]:
# Optional abundance-transition step: build a custom species set and recompute Abar/Zbar/Ye
if enable_abundance_transition:
    comp_transition = mesa_reader.prepare_abundance_transition(
        mmodel,
        species_names=abundance_target_species,
        closure_species=abundance_closure_species,
        missing_fill=abundance_missing_fill,
        )

    abundance_species_order = list(comp_transition['species'])
    abundance_profiles_mm = dict(comp_transition['abundances'])

    abar_mm_original = abar_mm.copy()
    zbar_mm_original = zbar_mm.copy()
    ye_mm_original = zbar_mm_original / np.maximum(abar_mm_original, 1e-99)

    abar_mm = np.asarray(comp_transition['Abar'], dtype=float)
    zbar_mm = np.asarray(comp_transition['Zbar'], dtype=float)
    ye_mm = np.asarray(comp_transition['Ye'], dtype=float)

    species_label = ', '.join(abundance_species_order)
    plotter.compare_composition_transition(
        r_mm,
        abar_mm_original,
        zbar_mm_original,
        ye_mm_original,
        abar_mm,
        zbar_mm,
        ye_mm,
        species_label=species_label,
        )

    print('Abundance transition enabled')
    print('Present species:', comp_transition['present_species'])
    print('Missing species (filled with zeros):', comp_transition['missing_species'])
    print('Closure species:', comp_transition['closure_species'])
else:
    ye_mm = zbar_mm / np.maximum(abar_mm, 1e-99)

# Rebuild composition interpolants on the same fine grid used for reintegration preparation
abar_mme = hse.grid2grid(r_mm, abar_mm, rmin, rmax, NrRESIZE)
zbar_mme = hse.grid2grid(r_mm, zbar_mm, rmin, rmax, NrRESIZE)

## Optional $\epsilon_{\rm nuc}(r)$ tweaking

In [ ]:
# Optional eps_nuc profile tweaking (shift + renormalization)
enuc_mm_modified = enuc_mm_original.copy()
enuc_tweak_summary = {
    'enabled': bool(enable_enuc_tweaking),
    'shift_dr_cm': float(enuc_shift_dr),
    'renorm_factor': float(enuc_renorm_factor),
}

if enable_enuc_tweaking:
    enuc_mm_modified = hse.tweak_enuc_profile(
        r_mm,
        enuc_mm_original,
        shift_dr=enuc_shift_dr,
        renorm_factor=enuc_renorm_factor,
        clip_non_negative=True,
    )

    print('eps_nuc tweaking is ENABLED (pre-reintegration diagnostics only).')
    print('Applied radius shift [cm]:', enuc_shift_dr)
    print('Applied renormalization factor:', enuc_renorm_factor)
else:
    print('eps_nuc tweaking is DISABLED. Using original MESA heating profile.')

# Profile used wherever enuc is requested for output/diagnostics
enuc_mm = enuc_mm_modified.copy()
enuc_inter = interp1d(r_mm, enuc_mm, bounds_error=False, fill_value='extrapolate')

# Luminosity check in the selected radial box
r_mm_box = r_mm[(r_mm >= rmin) & (r_mm <= rmax)]
enuc_mm_box_original = enuc_mm_original[(r_mm >= rmin) & (r_mm <= rmax)]
enuc_mm_box_modified = enuc_mm[(r_mm >= rmin) & (r_mm <= rmax)]

luminosity_mesa = cumulative_trapezoid(4.*np.pi*r_mm_box**2*enuc_mm_box_original, r_mm_box, initial=0.)
luminosity_new = cumulative_trapezoid(4.*np.pi*r_mm_box**2*enuc_mm_box_modified, r_mm_box, initial=0.)
print('LUMINOSITY ACCORDING TO MESA', luminosity_mesa[-1])
print('LUMINOSITY ACCORDING TO MODIFIED PROFILE', luminosity_new[-1])

if enable_enuc_tweaking and enuc_plot_compare:
    plotter.compare_profile(
        r_mm,
        enuc_mm_original,
        r_mm,
        enuc_mm,
        title=r'$\epsilon_{\rm nuc}$ comparison (MESA vs modified)',
        ylabel=r'$\epsilon_{\rm nuc}$',
        mesa_label=r'MESA $\epsilon_{\rm nuc}$',
        phl_label=r'Modified $\epsilon_{\rm nuc}$',
        xlim=(rmin/R_SOL*0.7, rmax/R_SOL*1.05),
        ylim=(0, 1.5*np.max(enuc_mm_box_original) if enuc_mm_box_original.size > 0 else 1.0),
    )


## Reintegration Input Preparation


In [ ]:
# Prepare for integration
press_p, temp_p, gravity_p, abar_p, zbar_p, nabladif_p, anchor_info = integrator.prepare_for_integration(
    integration_mode,
    use_non_uniform_composition,
    smoothing_width,
    smoothing_kernel,
    rmin,
    r_mme,
    gr_mme,
    unif_comp_mode,
    use_eos_p0,
    rho_mme,
    press_mme,
    temp_mme,
    abar_mme,
    zbar_mme,
    nabladif_mme,
    eos_mode=eos_mode,
    resize_factor=RESIZE_FACTOR,
    r_start=r_start,
    nabladif_zero_tol=nabladif_zero_tol,
    return_anchor_info=True,
    )
print('Preparation for integration done!')
print('Anchor diagnostics from preparation:', anchor_info)

# Use the snapped anchor radius so all plots show the true integration start.
plotter.r_integration_start = anchor_info['r_anchor']

plotter.compare_prepared_profiles(
    r_mm,
    r_mme,
    press_mm,
    press_p,
    temp_mm,
    temp_p,
    gr_mm,
    gravity_p,
    abar_mm,
    abar_p,
    zbar_mm,
    zbar_p,
    integration_mode,
    use_non_uniform_composition,
    nabladif_mesa=nabladif_mm,
    nabladif_prepared=nabladif_p,
    )

## Reintegration


In [ ]:
# Integration
if run_reintegration:
    p_hse_rk4, T_hse_rk4, rho_hse_rk4, r_hse_rk4, integration_info = integrator.hse_integrate(
        integration_mode,
        use_non_uniform_composition,
        press_p,
        temp_p,
        rmin,
        rmax,
        Nr,
        gravity_p,
        abar_p,
        zbar_p,
        nabladif_p,
        eos_mode=eos_mode,
        r_start=anchor_info['r_anchor'],
        return_diagnostics=True,
        )
    print('Integration done!')
    print('Integration diagnostics:', integration_info)

    # gravitational potential
    phi_rk4 = integrator.g_potential(gravity_p, r_hse_rk4)

    # Chemical info
    if use_non_uniform_composition:
        mu_phl = np.zeros((Nr))
        for i in range(Nr):
            mu_phl[i] = abar_p(r_hse_rk4[i])/(1 + zbar_p(r_hse_rk4[i]))
    else:
        print("abar_mid = ", abar_p)
        print("zbar_mid = ", zbar_p)
        mu_phl = abar_p/(1 + zbar_p)
        print("mu_phl = ", mu_phl)
else:
    print('Skipping reintegration: using preprocessed profiles only.')
    r_hse_rk4 = np.linspace(rmin, rmax, Nr)
    p_hse_rk4 = np.interp(r_hse_rk4, r_mme, press_mme)
    T_hse_rk4 = np.interp(r_hse_rk4, r_mme, temp_mme)
    rho_hse_rk4 = np.interp(r_hse_rk4, r_mme, rho_mme)
    integration_info = {'skipped': True, 'reason': 'run_reintegration=False'}
    phi_rk4 = integrator.g_potential(gravity_p, r_hse_rk4)
    if use_non_uniform_composition:
        mu_phl = abar_p(r_hse_rk4)/(1 + zbar_p(r_hse_rk4))
    else:
        mu_phl = abar_p/(1 + zbar_p)

## Run Summary


In [ ]:
if run_reintegration:
    # Build and print run summary diagnostics (independent of output_file toggle)
    run_summary, hse_residual = hse.build_run_summary(
        rmin=rmin,
        rmax=rmax,
        Nr=len(r_hse_rk4),
        p_hse=p_hse_rk4,
        rho_hse=rho_hse_rk4,
        r_hse=r_hse_rk4,
        gravity_fn=gravity_p,
        integration_mode=integration_mode,
        r_start=r_start,
        anchor_info=anchor_info,
        nabladif_zero_tol=nabladif_zero_tol,
        )

    # Keep legacy variable name used by visual diagnostics panels
    delta_hse = hse_residual

    print('Run summary:')
    print(json.dumps(run_summary, indent=2))
    print('HSE residual samples:', len(hse_residual))
else:
    print('Run summary: integration skipped, no diagnostics available.')
    delta_hse = None

## Output Writing


In [ ]:
# Output writing (.in + .npz) delegated to backend helper
output_result = hse.write_output_bundle(
    output_file=output_file,
    name_of_in_file=name_of_in_file,
    r_hse_rk4=r_hse_rk4,
    gravity_p=gravity_p,
    phi_rk4=phi_rk4,
    p_hse_rk4=p_hse_rk4,
    rho_hse_rk4=rho_hse_rk4,
    T_hse_rk4=T_hse_rk4,
    r_mm=r_mm,
    kappa_mm=kappa_mm,
    use_non_uniform_composition=use_non_uniform_composition,
    abar_p=abar_p,
    zbar_p=zbar_p,
    enuc_inter=enuc_inter,
    enuc_at_output=enuc_at_output,
    abundance_profiles_mm=abundance_profiles_mm,
    output_column_names_requested=output_column_names_requested,
    build_output_table_fn=hse.build_output_table,
    rmin=rmin,
    rmax=rmax,
    integration_mode=integration_mode,
    r_start=r_start,
    anchor_info=anchor_info,
    nabladif_zero_tol=nabladif_zero_tol,
    build_run_summary_fn=hse.build_run_summary,
    integration_info=integration_info,
    mesa_path=mesa_path,
    helm_table_path=helm_table_path,
    hse_library_pointer=locals().get('hse_library_pointer', None),
    mesa_reader_pointer=mesa_reader_pointer,
    run_reintegration=run_reintegration,
    use_eos_p0=use_eos_p0,
    uniform_composition_strategy=uniform_composition_strategy,
    eos_mode=eos_mode,
    missing_profile_policy=missing_profile_policy,
    press_mm=press_mm,
    temp_mm=temp_mm,
    rho_mm=rho_mm,
    gr_mm=gr_mm,
    abar_mm=abar_mm,
    zbar_mm=zbar_mm,
    ye_mm=ye_mm,
    nabladif_mm=nabladif_mm,
    r_mme=r_mme,
    press_mme=press_mme,
    temp_mme=temp_mme,
    rho_mme=rho_mme,
    gr_mme=gr_mme,
    abar_mme=abar_mme,
    zbar_mme=zbar_mme,
    nabladif_mme=nabladif_mme,
    smoothing_width=smoothing_width,
    smoothing_kernel=smoothing_kernel,
    resize_factor=RESIZE_FACTOR,
    notebook_path='create_input_library/create_input.ipynb',
)

if output_result['written']:
    output_columns = output_result['output_columns']
    output_table = output_result['output_table']
    hse_residual = output_result['hse_residual']
    abundance_profiles_out = output_result['abundance_profiles_out']

## Visual Diagnostics


In [ ]:
# Visual Check: render split multiplots (linear + log side-by-side)

# Composition profiles on the RK4 grid (needed by multiple diagnostics).
if use_non_uniform_composition:
    abar_phl = np.asarray(abar_p(r_hse_rk4), dtype=float)
    zbar_phl = np.asarray(zbar_p(r_hse_rk4), dtype=float)
else:
    abar_phl = np.full_like(r_hse_rk4, float(abar_p), dtype=float)
    zbar_phl = np.full_like(r_hse_rk4, float(zbar_p), dtype=float)

# Build EOS diagnostics on the RK4 grid via backend helper.
eos_profiles = hse.compute_eos_profiles(
    rho_hse_rk4,
    T_hse_rk4,
    abar_phl,
    zbar_phl,
    fields=['s', 'nabla_ad', 'gamma_1', 'gamma_2', 'gamma_3', 'cp', 'cv', 'e', 'delta', 'phi', 'sound'],
    eos_mode=eos_mode,
    dtype=eos_output_dtype,
    eos_table=eos_table,
)

entropy_phl = np.asarray(eos_profiles['s'], dtype=float)
nabla_ad_rk4 = np.asarray(eos_profiles['nabla_ad'], dtype=float)
gamma1_rk4 = np.asarray(eos_profiles['gamma_1'], dtype=float)
gamma2_rk4 = np.asarray(eos_profiles['gamma_2'], dtype=float)
gamma3_rk4 = np.asarray(eos_profiles['gamma_3'], dtype=float)
cp_rk4 = np.asarray(eos_profiles['cp'], dtype=float)
cv_rk4 = np.asarray(eos_profiles['cv'], dtype=float)
eint_rk4 = np.asarray(eos_profiles['e'], dtype=float)
delta_rk4 = np.asarray(eos_profiles['delta'], dtype=float)
denom_alpha = np.asarray(cp_rk4 - cv_rk4, dtype=float)
alpha_rk4 = np.where(
    np.abs(denom_alpha) > 1e-99,
    np.asarray(p_hse_rk4, dtype=float) * np.asarray(delta_rk4, dtype=float)**2 / (np.asarray(rho_hse_rk4, dtype=float) * np.asarray(T_hse_rk4, dtype=float) * denom_alpha),
    np.nan,
)
phi_thermo_rk4 = np.asarray(eos_profiles['phi'], dtype=float)
sound_rk4 = np.asarray(eos_profiles['sound'], dtype=float)

# Thermodynamic gradients reconstructed from post-integration profiles.
lnP_rk4 = np.log(np.clip(np.asarray(p_hse_rk4, dtype=float), 1e-99, None))
lnT_rk4 = np.log(np.clip(np.asarray(T_hse_rk4, dtype=float), 1e-99, None))
nabla_post = np.gradient(lnT_rk4, lnP_rk4, edge_order=1)
nabladif_post = nabla_post - nabla_ad_rk4

# Mu-gradient diagnostics: use loaded MESA profile when available, else derive.
nabla_mu_mm = _profile('nabla_mu')
if not np.any(np.isfinite(nabla_mu_mm)):
    mu_mm = abar_mm / np.maximum(1.0 + zbar_mm, 1e-99)
    lnP_mm = np.log(np.clip(np.asarray(press_mm, dtype=float), 1e-99, None))
    lnmu_mm = np.log(np.clip(np.asarray(mu_mm, dtype=float), 1e-99, None))
    nabla_mu_mm = np.gradient(lnmu_mm, lnP_mm, edge_order=1)

if np.ndim(mu_phl) == 0:
    mu_phl_profile = np.full_like(r_hse_rk4, float(mu_phl), dtype=float)
else:
    mu_phl_profile = np.asarray(mu_phl, dtype=float)
lnmu_phl = np.log(np.clip(mu_phl_profile, 1e-99, None))
nabla_mu_phl = np.gradient(lnmu_phl, lnP_rk4, edge_order=1)

# Optional MESA fields with graceful fallback to NaN arrays.
gamma2_mm = _profile('gamma2')
phi_thermo_mm = _profile('phi_thermo')

# Gravitational potential comparison in the same [rmin, rmax] box.
phi_mm = _profile('phi')
phi_mask = (r_mm >= rmin) & (r_mm <= rmax)
r_mm_box = r_mm[phi_mask]
if not np.any(np.isfinite(phi_mm)):
    gravity_mm = interp1d(r_mm, gr_mm, bounds_error=False, fill_value='extrapolate')
    phi_mm = integrator.g_potential(gravity_mm, r_mm_box)
else:
    phi_mm = np.asarray(phi_mm[phi_mask], dtype=float)

# Relative energy-difference diagnostic on the RK4 grid.
energy_mm_on_rk4 = np.interp(r_hse_rk4, r_mm, energy_mm)
denom_energy = np.where(np.abs(energy_mm_on_rk4) > 1e-99, energy_mm_on_rk4, np.nan)
diff_energy = (energy_mm_on_rk4 - eint_rk4) / denom_energy

# Entropy alignment option for shape-only comparison.
entropy_mm_centered = entropy_mm.copy()
entropy_phl_centered = entropy_phl.copy()
if entropy_alignment_mode == 'anchor':
    r_anchor = float(r_hse_rk4[0])
    entropy_shift = np.interp(r_anchor, r_mm, entropy_mm) - np.interp(r_anchor, r_hse_rk4, entropy_phl)
    entropy_mm_centered = entropy_mm - entropy_shift
elif entropy_alignment_mode != 'none':
    raise ValueError(f"Unknown entropy_alignment_mode: {entropy_alignment_mode!r}")

# Base definitions (one entry per variable, later expanded to linear/log pairs in the library)
base_panels = [
    dict(key='pressure', x_mesa=r_mm, y_mesa=press_mm, x_phl=r_hse_rk4, y_phl=p_hse_rk4, title='$P$', ylabel='$P$', plot_every=plot_every, mesa_label='MESA $P$', phl_label='RK4 $P$'),
    dict(key='temperature', x_mesa=r_mm, y_mesa=temp_mm, x_phl=r_hse_rk4, y_phl=T_hse_rk4, title='$T$', ylabel='$T$', plot_every=plot_every, mesa_label='MESA $T$', phl_label='RK4 $T$'),
    dict(key='density', x_mesa=r_mm, y_mesa=rho_mm, x_phl=r_hse_rk4, y_phl=rho_hse_rk4, title=r'$\rho$', ylabel=r'$\rho$', plot_every=plot_every, mesa_label=r'MESA $\rho$', phl_label=r'RK4 $\rho$'),
    dict(key='entropy', x_mesa=r_mm, y_mesa=entropy_mm_centered, x_phl=r_hse_rk4, y_phl=entropy_phl_centered, title='$s$', ylabel='$s$', plot_every=plot_every, mesa_label='MESA $s$', phl_label='RK4 $s$'),
    dict(key='opacity', x_mesa=r_mm, y_mesa=kappa_mm, x_phl=r_hse_rk4, y_phl=kappa, title=r'$\kappa$', ylabel=r'$\kappa$', plot_every=plot_every, mesa_label=r'MESA $\kappa$', phl_label=r'RK4 $\kappa$'),
    dict(key='gravity', x_mesa=r_mm, y_mesa=gr_mm, x_phl=r_hse_rk4, y_phl=gravity_p(r_hse_rk4), title='$g$', ylabel='$g$', plot_every=plot_every, mesa_label='MESA $g$', phl_label='RK4 $g$'),
    dict(key='hse_residual', x_mesa=r_hse_rk4[1:-1], y_mesa=delta_hse, title='HSE residual', ylabel=r'$\delta_{\rm HSE}$', plot_every=plot_every, mesa_label=r'RK4 $\delta_{\rm HSE}$'),
    dict(key='abar', x_mesa=r_mm, y_mesa=abar_mm, x_phl=r_hse_rk4, y_phl=abar_phl, title=r'$\bar{A}$', ylabel=r'$\bar{A}$', plot_every=plot_every, mesa_label=r'MESA $\bar{A}$', phl_label=r'RK4 $\bar{A}$'),
    dict(key='zbar', x_mesa=r_mm, y_mesa=zbar_mm, x_phl=r_hse_rk4, y_phl=zbar_phl, title=r'$\bar{Z}$', ylabel=r'$\bar{Z}$', plot_every=plot_every, mesa_label=r'MESA $\bar{Z}$', phl_label=r'RK4 $\bar{Z}$'),
    dict(key='nabla_ad', x_mesa=r_mm, y_mesa=nabla_ad_mm, x_phl=r_hse_rk4, y_phl=nabla_ad_rk4, title=r'$\nabla_{\rm ad}$', ylabel=r'$\nabla_{\rm ad}$', plot_every=plot_every, mesa_label=r'MESA $\nabla_{\rm ad}$', phl_label=r'RK4 $\nabla_{\rm ad}$'),
    dict(key='nabladiff', x_mesa=r_mm, y_mesa=nabladif_mm, x_phl=r_hse_rk4, y_phl=nabladif_post, title=r'$\nabla-\nabla_{\rm ad}$', ylabel=r'$\nabla-\nabla_{\rm ad}$', plot_every=plot_every, mesa_label=r'MESA $\nabla-\nabla_{\rm ad}$', phl_label=r'RK4 $\nabla-\nabla_{\rm ad}$'),
    dict(key='nabla', x_mesa=r_mm, y_mesa=nabla_mm, x_phl=r_hse_rk4, y_phl=nabla_post, title=r'$\nabla$', ylabel=r'$\nabla$', plot_every=plot_every, mesa_label=r'MESA $\nabla$', phl_label=r'RK4 $\nabla$'),
    dict(key='nabla_mu', x_mesa=r_mm, y_mesa=nabla_mu_mm, x_phl=r_hse_rk4, y_phl=nabla_mu_phl, title=r'$\nabla_\mu$', ylabel=r'$\nabla_\mu$', plot_every=plot_every, mesa_label=r'MESA $\nabla_\mu$', phl_label=r'RK4 $\nabla_\mu$'),
    dict(key='gamma1', x_mesa=r_mm, y_mesa=gamma1_mm, x_phl=r_hse_rk4, y_phl=gamma1_rk4, title=r'$\Gamma_1$', ylabel=r'$\Gamma_1$', plot_every=plot_every, mesa_label=r'MESA $\Gamma_1$', phl_label=r'RK4 $\Gamma_1$'),
    dict(key='gamma2', x_mesa=r_mm, y_mesa=gamma2_mm, x_phl=r_hse_rk4, y_phl=gamma2_rk4, title=r'$\Gamma_2$', ylabel=r'$\Gamma_2$', plot_every=plot_every, mesa_label=r'MESA $\Gamma_2$', phl_label=r'RK4 $\Gamma_2$'),
    dict(key='gamma3', x_mesa=r_mm, y_mesa=gamma3_mm, x_phl=r_hse_rk4, y_phl=gamma3_rk4, title=r'$\Gamma_3$', ylabel=r'$\Gamma_3$', plot_every=plot_every, mesa_label=r'MESA $\Gamma_3$', phl_label=r'RK4 $\Gamma_3$'),
    dict(key='cp', x_mesa=r_mm, y_mesa=cp_mm, x_phl=r_hse_rk4, y_phl=cp_rk4, title='$c_p$', ylabel='$c_p$', plot_every=plot_every, mesa_label='MESA $c_p$', phl_label='RK4 $c_p$'),
    dict(key='cv', x_mesa=r_mm, y_mesa=cv_mm, x_phl=r_hse_rk4, y_phl=cv_rk4, title='$c_v$', ylabel='$c_v$', plot_every=plot_every, mesa_label='MESA $c_v$', phl_label='RK4 $c_v$'),
    dict(key='phi', x_mesa=r_mm_box, y_mesa=phi_mm, x_phl=r_hse_rk4, y_phl=phi_rk4, title=r'$\Phi$', ylabel=r'$\Phi$', plot_every=plot_every, mesa_label=r'MESA $\Phi$', phl_label=r'RK4 $\Phi$'),
    dict(key='energy', x_mesa=r_mm, y_mesa=energy_mm, x_phl=r_hse_rk4, y_phl=eint_rk4, title='$e$', ylabel='$e$', plot_every=plot_every, mesa_label='MESA $e$', phl_label='RK4 $e$'),
    dict(key='energy_diff', x_mesa=r_hse_rk4, y_mesa=diff_energy, title='Relative energy difference', ylabel=r'$(e_{\rm MESA}-e_{\rm RK4})/e_{\rm MESA}$', plot_every=plot_every, mesa_label=r'$(e_{\rm MESA}-e_{\rm RK4})/e_{\rm MESA}$'),
    dict(key='delta', x_mesa=r_mm, y_mesa=delta_mm, x_phl=r_hse_rk4, y_phl=delta_rk4, title=r'$\delta$', ylabel=r'$\delta$', plot_every=plot_every, mesa_label=r'MESA $\delta$', phl_label=r'RK4 $\delta$'),
    dict(key='alpha', x_mesa=r_mm, y_mesa=alpha_mm, x_phl=r_hse_rk4, y_phl=alpha_rk4, title=r'$\alpha$', ylabel=r'$\alpha$', plot_every=plot_every, mesa_label=r'MESA $\alpha$', phl_label=r'RK4 $\alpha$'),
    dict(key='phi_thermo', x_mesa=r_mm, y_mesa=phi_thermo_mm, x_phl=r_hse_rk4, y_phl=phi_thermo_rk4, title=r'$\phi_{\rm thermo}$', ylabel=r'$\phi_{\rm thermo}$', plot_every=plot_every, mesa_label=r'MESA $\phi_{\rm thermo}$ (Helm EOS)', phl_label=r'RK4 $\phi_{\rm thermo}$'),
    dict(key='sound', x_mesa=r_mm, y_mesa=cs_mm, x_phl=r_hse_rk4, y_phl=sound_rk4, title='$c_s$', ylabel='$c_s$', plot_every=plot_every, mesa_label='MESA $c_s$', phl_label='RK4 $c_s$'),
]

# Simple mode switch:
# - run_reintegration=True  -> MESA vs RK4
# - run_reintegration=False -> MESA only
if not run_reintegration:
    mesa_only_exclude = {'hse_residual', 'energy_diff'}
    base_panels = [p for p in base_panels if p.get('key') not in mesa_only_exclude]
    for p in base_panels:
        p.pop('x_phl', None)
        p.pop('y_phl', None)
        p.pop('phl_label', None)

# Figure 1 primary variables are configured in the Toggles section.
primary_panel_multiplier = 2 if figure1_scale_mode == 'split' else 1
primary_panel_count = max(1, primary_panel_multiplier * len(primary_keys))

(fig_primary, axs_primary), (fig_secondary, axs_secondary) = plotter.plot_visual_check_split_figures(
    base_panels=base_panels,
    primary_keys=primary_keys,
    figure1_scale_mode=figure1_scale_mode,
    ncols=2,
    primary_figsize=(10, 2 * primary_panel_count),
    secondary_figsize=(10, 2 * max(1, len(base_panels) - len(primary_keys))),
    show_secondary_legends=figure2_show_legends,
    )